# Strategy — exercises

Run `example.ipynb` first. Each exercise below re-declares the pieces it needs. Complete the code cells; do not look up a solution until you have tried the exercise.


## Exercise 1 — Add a concrete strategy (basic application)

**Goal:** Add `PickupShipping` that always charges `0.0`, without changing `Checkout` or the existing shipping classes.

**Acceptance criteria:**
- `PickupShipping` implements `ShippingStrategy`.
- `Checkout(PickupShipping()).total(40.0, 2.0)` prints `40.0`.
- No existing class is edited — only the new strategy is added, then used.


In [ ]:
from abc import ABC, abstractmethod


class ShippingStrategy(ABC):
    @abstractmethod
    def cost(self, weight_kg: float) -> float:
        ...


class StandardShipping(ShippingStrategy):
    def cost(self, weight_kg: float) -> float:
        return 5.0 + 1.25 * weight_kg

class PickupShipping(ShippingStrategy):
    def cost(self, weight_kg: float) -> float:
        return 0.0


class Checkout:
    def __init__(self, shipping: ShippingStrategy) -> None:
        self._shipping = shipping

    def set_shipping(self, shipping: ShippingStrategy) -> None:
        self._shipping = shipping

    def total(self, subtotal: float, weight_kg: float) -> float:
        return round(subtotal + self._shipping.cost(weight_kg), 2)


# Your solution here
checkout = Checkout(PickupShipping())
print(checkout.total(40.0, 2.0))  # 40.0


## Exercise 2 — Swap the strategy at runtime (adaptation)

**Goal:** Keep `Checkout` free of shipping `if/elif` branches. Write a *client-side* helper `select_shipping(weight_kg)` that returns `StandardShipping` when `weight_kg <= 5`, otherwise `ExpressShipping`. Create one `Checkout`, then compute totals for 2 kg and 8 kg by calling `set_shipping(...)` with the helper's result.

**Acceptance criteria:**
- `Checkout` is not modified (no new methods, no conditionals inside it).
- 2 kg uses Standard and 8 kg uses Express.
- Printed totals for subtotal `40.0` are `47.5` then `72.0`.


In [ ]:
from abc import ABC, abstractmethod


class ShippingStrategy(ABC):
    @abstractmethod
    def cost(self, weight_kg: float) -> float:
        ...


class StandardShipping(ShippingStrategy):
    def cost(self, weight_kg: float) -> float:
        return 5.0 + 1.25 * weight_kg


class ExpressShipping(ShippingStrategy):
    def cost(self, weight_kg: float) -> float:
        return 12.0 + 2.5 * weight_kg


class Checkout:
    def __init__(self, shipping: ShippingStrategy) -> None:
        self._shipping = shipping

    def set_shipping(self, shipping: ShippingStrategy) -> None:
        self._shipping = shipping

    def total(self, subtotal: float, weight_kg: float) -> float:
        return round(subtotal + self._shipping.cost(weight_kg), 2)


# Your solution here
def select_shipping(weight: float) -> ShippingStrategy:
    if weight <= 5.0: 
        return StandardShipping()
    else:
        return ExpressShipping()


checkout = Checkout(StandardShipping())

checkout.set_shipping(select_shipping(2.0))
print(checkout.total(40.0, 2.0))  # 47.5

checkout.set_shipping(select_shipping(8.0))
print(checkout.total(40.0, 8.0))  # 72.0

## Exercise 3 — Critique (design judgment)

In a markdown cell below, answer:

1. A teammate says Strategy and State are the same because both swap objects that change behavior. How would you tell them apart using `Checkout` vs a package-tracking object whose status moves `ordered -> shipped -> delivered`?
2. Python lets you pass a function instead of a strategy class (`checkout.total` could call `shipping_fn(weight)`). When is a class-based Strategy still worth it, and when is a function enough?
3. Is Strategy always the right replacement for a chain of `if/elif`? Give one case where you would keep the conditionals.


1. State only changes the object, and might give them different behaviour, while strategy changes an ALGORITHM, depending on what is needed. 

2. When the changes aren't that big, or is just one variation.

3. Keep if/elif when there are only a few simple, stable cases, such as choosing free shipping above a minimum order amount and standard shipping otherwise. Use Strategy when the alternatives are substantial algorithms that vary independently or are likely to grow.
